# 🏭 Project 23 · Assembly Line Intelligence
## REINFORCE Policy Gradient for Production Decision Optimization

**Reinforcement Learning Family | Operational Excellence Portfolio | LozanoLsa**

---

Q-Learning asked: *what is the expected value of this action in this state?*
It answered by maintaining a lookup table — one entry per state-action pair.

Policy Optimization asks a more direct question: *given this state, what should I do?*
Not via a table. Via a **function** — a parameterized policy that maps every state
directly to a probability distribution over actions.

$$\pi_\theta(a \mid s) = \text{softmax}(W_\theta \cdot s + b_\theta)$$

The parameters $\theta = \{W, b\}$ are updated by **gradient ascent** on expected reward:

$$\theta \leftarrow \theta + \alpha \cdot G_t \cdot \nabla_\theta \log \pi_\theta(a_t \mid s_t)$$

This is **REINFORCE** — the foundational policy gradient algorithm. No value table needed.
The policy learns directly from the consequences of its own decisions.

---

## The Industrial Problem: Assembly Line Decision Engine

A 6-station assembly line requires continuous operational decisions.
At each step, the line manager (the agent) observes 11 state variables
and selects one of 6 possible interventions:

| Action | Intervention | Primary Effect |
|--------|-------------|---------------|
| 0 | Reassign operator | ↑ operator efficiency, ↓ micro-stops |
| 1 | Increase speed | ↑ throughput, ↑ failure risk |
| 2 | Decrease speed | ↓ throughput, ↓ failure risk |
| 3 | Quick maintenance | ↓↓ micro-stops, ↓ failure prob |
| 4 | Redirect flow | ↓ WIP, ↑ speed |
| 5 | No action | No change (baseline) |

A **random manager** (baseline) averages 94.9 reward per step.
A **trained REINFORCE agent** (400 episodes) reaches 127.3 reward per step —
a **+34% improvement** through learned prioritization of flow redirection and operator management.

The policy doesn't follow fixed rules. It learns a probabilistic strategy,
calibrated to the actual dynamics of the assembly line.

---
**LozanoLsa | Operational Excellence · MBB · Machine Learning | GitHub: LozanoLsa**


## 2 · Setup

In [ ]:
# ── 2 · Setup ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor":   "#161b22",
    "axes.edgecolor":   "#30363d",
    "axes.labelcolor":  "#c9d1d9",
    "xtick.color":      "#8b949e",
    "ytick.color":      "#8b949e",
    "text.color":       "#c9d1d9",
    "grid.color":       "#21262d",
    "grid.linestyle":   "--",
    "grid.alpha":       0.4,
    "legend.facecolor": "#161b22",
    "legend.edgecolor": "#30363d",
})

C_BLUE = "#4C9BE8"
C_RED  = "#E8574C"
C_OK   = "#3FB950"
C_WARN = "#F5A623"
C_BG   = "#0d1117"
C_CARD = "#161b22"

ACTION_NAMES = [
    "Reassign Operator", "Increase Speed", "Decrease Speed",
    "Quick Maintenance", "Redirect Flow", "No Action"
]

print("REINFORCE Policy Gradient pipeline ready.")
print("  State space : 11 continuous features")
print("  Action space: 6 discrete interventions")
print("  Algorithm   : REINFORCE (Monte Carlo Policy Gradient)")

## 3 · Load Data — Random Policy Behavioral Dataset

Before training an optimized policy, we analyze the **behavioral baseline**:
23,832 steps collected from a purely random manager (300 episodes, 80 steps each).

This dataset answers: *what does the assembly line look like under unguided decision-making?*
It establishes the performance floor that the REINFORCE agent must exceed.

| Column | Description |
|--------|-------------|
| `ct_norm_s1–s6` | Normalized cycle time per station [0=fast, 1=slow] |
| `wip_norm` | WIP level, normalized [0=empty, 1=saturated] |
| `speed_norm` | Line speed, normalized [0=stopped, 1=max] |
| `failure_prob` | Estimated failure probability [0–0.25] |
| `micro_stops_norm` | Micro-stop accumulation, normalized |
| `operator_efficiency` | Operator performance [0.60–1.00] |
| `action` | Decision taken [0–5] |
| `reward` | Step reward (composite: throughput − WIP − micro-stops − failure) |
| `throughput_uph` | Units per hour at this step |
| `episode` / `step` | Trajectory position |


In [ ]:
try:
    df = pd.read_csv("Data_people.csv")
except FileNotFoundError:
    df = pd.read_csv(
        "https://raw.githubusercontent.com/LozanoLsa/"
        "PolicyOpt_Assembly/main/Data_people.csv"
    )

features = [f"ct_norm_s{i}" for i in range(1, 7)] + [
    "wip_norm", "speed_norm", "failure_prob",
    "micro_stops_norm", "operator_efficiency"
]

print(f"Dataset: {df.shape[0]:,} steps | {df['episode'].nunique()} episodes")
print(f"Mean step reward    : {df['reward'].mean():.2f}")
print(f"Mean throughput UPH : {df['throughput_uph'].mean():.2f}")
print()
print("Action distribution (random policy — uniform expected):")
print(df["action"].value_counts().sort_index()
      .rename(dict(enumerate(ACTION_NAMES))).to_string())
df.head(8)

## 4 · Exploratory Data Analysis

Three views of the random baseline behavior:

1. **Reward distribution** — what range of outcomes does random decision-making produce?
2. **Throughput over time** — does the line drift without guidance?
3. **State correlations** — which variables co-move, revealing system interdependencies?


In [ ]:
# ── EDA 1: Reward + Throughput distributions ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.patch.set_facecolor(C_BG)
fig.suptitle("Random Policy Behavioral Data — 23,832 Steps", color="white", fontsize=13)

# Reward distribution
ax = axes[0]
ax.hist(df["reward"], bins=60, color=C_BLUE, edgecolor=C_BG, alpha=0.85)
ax.axvline(df["reward"].mean(), color=C_RED, lw=2, ls="--", label=f"Mean={df['reward'].mean():.1f}")
ax.set_xlabel("Step Reward"); ax.set_ylabel("Count")
ax.set_title("Reward Distribution", color="white"); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Throughput distribution
ax = axes[1]
ax.hist(df["throughput_uph"], bins=60, color=C_OK, edgecolor=C_BG, alpha=0.85)
ax.axvline(df["throughput_uph"].mean(), color=C_RED, lw=2, ls="--",
           label=f"Mean={df['throughput_uph'].mean():.1f}")
ax.set_xlabel("Throughput (UPH)"); ax.set_ylabel("Count")
ax.set_title("Throughput Distribution", color="white"); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Action frequency
ax = axes[2]
act_counts = df["action"].value_counts().sort_index()
ax.bar(range(6), act_counts.values, color=C_BLUE, edgecolor=C_BG, alpha=0.85)
ax.set_xticks(range(6))
ax.set_xticklabels(["Reassign","Incr.Spd","Decr.Spd","Maint.","Redirect","NoAct"],
                   rotation=20, ha="right", fontsize=8)
ax.set_ylabel("Frequency"); ax.set_title("Action Frequency (Random)", color="white")
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("fig_01_eda.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()
print("Random policy: uniform action distribution, wide reward variance.")
print("This is the baseline the REINFORCE agent must surpass.")

In [ ]:
# ── EDA 2: Average episode reward + throughput trajectories ──────────────────
ep_stats = df.groupby("episode")[["reward", "throughput_uph"]].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
fig.patch.set_facecolor(C_BG)

axes[0].plot(ep_stats.index, ep_stats["reward"], color=C_BLUE, lw=0.8, alpha=0.7)
axes[0].plot(ep_stats["reward"].rolling(30).mean(), color=C_OK, lw=2, label="30-ep mean")
axes[0].axhline(ep_stats["reward"].mean(), color=C_RED, lw=1.5, ls="--",
                label=f"Overall mean = {ep_stats['reward'].mean():.1f}")
axes[0].set_xlabel("Episode"); axes[0].set_ylabel("Mean Reward / Step")
axes[0].set_title("Per-Episode Mean Reward — Random Policy", color="white")
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_stats.index, ep_stats["throughput_uph"], color=C_WARN, lw=0.8, alpha=0.7)
axes[1].plot(ep_stats["throughput_uph"].rolling(30).mean(), color=C_OK, lw=2)
axes[1].set_xlabel("Episode"); axes[1].set_ylabel("Mean Throughput (UPH)")
axes[1].set_title("Per-Episode Mean Throughput — Random Policy", color="white")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fig_02_baselines.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()
print("No learning trend in random policy — performance stays flat across 300 episodes.")
print("This flatness is what REINFORCE breaks through.")

In [ ]:
# ── EDA 3: Correlation between state variables and reward ─────────────────────
import seaborn as sns

corr_cols = features + ["reward", "throughput_uph"]
corr = df[corr_cols].corr()["reward"].drop("reward").sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor(C_BG)
colors = [C_RED if v < 0 else C_OK for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor=C_BG, height=0.55)
ax.axvline(0, color="#8b949e", lw=1)
ax.set_xlabel("Pearson r with Step Reward", fontsize=10)
ax.set_title("State Feature Correlation with Reward", color="white", fontsize=12)
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("fig_03_corr.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()
print("operator_efficiency and speed_norm positively drive reward.")
print("failure_prob and wip_norm negatively impact reward — the agent must learn to suppress these.")

## 5 · Environment & Policy Architecture

### The Environment (Inline — No External Import)

The `AssemblyLineEnv` is defined directly here for portability and transparency.
State dimension: **11** (6 normalized cycle times + 5 process variables).

### The Policy: Softmax Over Linear Features

$$\pi_\theta(a \mid s) = \frac{\exp(W_a \cdot s + b_a)}{\sum_{a'} \exp(W_{a'} \cdot s + b_{a'})}$$

Parameters $\theta = \{W \in \mathbb{R}^{6 \times 11}, b \in \mathbb{R}^6\}$ initialized to zero.
At initialization, all actions are equally likely — the policy knows nothing.

The gradient of the log-policy with respect to $W_a$:

$$\nabla_{W_a} \log \pi_\theta(a \mid s) = (\mathbf{1}[a=a_t] - \pi_\theta(a \mid s)) \cdot s^\top$$

This is the signal that tells each action weight: *did choosing this action lead to good outcomes?*


In [ ]:
# ── AssemblyLineEnv (self-contained) ─────────────────────────────────────────
class AssemblyLineEnv:
    def __init__(self, n_stations=6):
        self.n = n_stations
        self.action_space = 6
        self.ct_min, self.ct_max = 20, 45
        self.wip_max, self.vel_max, self.mic_max = 40, 1.2, 15

    def _norm(self, x, lo, hi): return (x - lo) / (hi - lo)

    def reset(self):
        self.ct    = np.random.uniform(28, 35, self.n)
        self.wip   = np.random.randint(6, 15)
        self.vel   = np.random.uniform(0.80, 1.00)
        self.p_f   = np.random.uniform(0.05, 0.12)
        self.micro = np.random.randint(0, 5)
        self.efic  = np.random.uniform(0.75, 0.95)
        return self._state()

    def _state(self):
        ct_n = [self._norm(c, self.ct_min, self.ct_max) for c in self.ct]
        return np.array(ct_n + [self.wip / self.wip_max, self.vel / self.vel_max,
                                  self.p_f, self.micro / self.mic_max, self.efic],
                        dtype=np.float32)

    def step(self, action):
        if   action == 0: self.efic += np.random.uniform(0, 0.05); self.micro = max(0, self.micro-1)
        elif action == 1: self.vel += 0.03; self.ct -= np.random.uniform(0.3, 0.8, self.n); self.p_f += 0.02
        elif action == 2: self.vel -= 0.03; self.ct += np.random.uniform(0.5, 1.2, self.n); self.p_f -= 0.015
        elif action == 3: self.micro = max(0, self.micro-3); self.p_f -= 0.03; self.ct += np.random.uniform(0.2, 0.6, self.n)
        elif action == 4: self.wip -= np.random.randint(1, 4); self.vel += 0.01

        self.ct    += np.random.uniform(-0.5, 0.5, self.n)
        self.wip   += np.random.randint(-1, 2)
        self.micro += np.random.randint(0, 2)
        self.p_f   += np.random.uniform(-0.01, 0.01)
        self.ct     = np.clip(self.ct,    self.ct_min, self.ct_max)
        self.wip    = np.clip(self.wip,   0,           self.wip_max)
        self.vel    = np.clip(self.vel,   0.5,         self.vel_max)
        self.p_f    = np.clip(self.p_f,   0.0,         0.25)
        self.micro  = np.clip(self.micro, 0,           self.mic_max)
        self.efic   = np.clip(self.efic,  0.60,        1.00)

        avg_ct     = np.mean(self.ct)
        throughput = (3600 / avg_ct) * self.vel * self.efic
        reward     = 1.2*throughput - 4.0*self.micro - 1.5*self.wip - 10.0*self.p_f

        done = self.wip >= self.wip_max or self.p_f > 0.20
        if done: reward -= 200

        return self._state(), reward, done, {"throughput": throughput}

env = AssemblyLineEnv()
state_dim = env.reset().shape[0]
print(f"Environment ready. State dim: {state_dim}  |  Actions: {env.action_space}")

# Sanity check: one random step
s = env.reset()
ns, r, done, info = env.step(1)
print(f"Sample step → reward={r:.2f}, throughput={info['throughput']:.1f}, done={done}")

In [ ]:
# ── SoftmaxPolicy ────────────────────────────────────────────────────────────
class SoftmaxPolicy:
    def __init__(self, state_dim, n_actions, lr=0.002):
        self.W  = np.zeros((n_actions, state_dim))
        self.b  = np.zeros(n_actions)
        self.lr = lr

    def probs(self, state):
        logits = self.W @ state + self.b
        logits -= logits.max()           # numerical stability
        exp = np.exp(logits)
        return exp / exp.sum()

    def select_action(self, state):
        return int(np.random.choice(len(self.b), p=self.probs(state)))

    def update(self, states, actions, returns):
        for s, a, G in zip(states, actions, returns):
            p = self.probs(s)
            I = np.zeros(len(self.b)); I[a] = 1.0
            dlog  = np.outer(I - p, s)   # gradient of log pi w.r.t. W
            self.W += self.lr * G * dlog
            self.b += self.lr * G * (I - p)

# Initialize policy
policy = SoftmaxPolicy(state_dim=state_dim, n_actions=env.action_space, lr=0.002)
print("SoftmaxPolicy initialized. All action probabilities = 1/6 at start.")
test_s = env.reset()
print(f"Initial action probs: {policy.probs(test_s).round(3)}")

## 6 · REINFORCE Algorithm

### Monte Carlo Policy Gradient

At the end of each episode, the agent computes **discounted returns**:

$$G_t = \sum_{k=0}^{T-t} \gamma^k r_{t+k}$$

Returns are **normalized** (zero-mean, unit variance) for training stability —
this is standard practice that reduces variance without introducing bias.

Each $(s_t, a_t, G_t)$ triple becomes a training sample:
"When I was in state $s_t$ and chose action $a_t$, the downstream return was $G_t$."

High $G_t$ → increase probability of $a_t$ in state $s_t$.
Low $G_t$ → decrease it.

### Hyperparameters

| Parameter | Value | Role |
|-----------|-------|------|
| `gamma` | 0.99 | Discount factor — future rewards nearly as valuable as immediate |
| `lr` | 0.002 | Learning rate — small steps for stable gradient ascent |
| `episodes` | 400 | Training budget |
| `max_steps` | 100 | Episode length cap |


In [ ]:
# ── REINFORCE training loop ───────────────────────────────────────────────────
def reinforce(env, policy, episodes=400, gamma=0.99, max_steps=100):
    episode_rewards = []
    np.random.seed(42)

    for ep in range(episodes):
        state = env.reset()
        states, actions, rewards = [], [], []

        for _ in range(max_steps):
            action = policy.select_action(state)
            next_state, reward, done, _ = env.step(action)
            states.append(state); actions.append(action); rewards.append(reward)
            state = next_state
            if done:
                break

        # Discounted returns
        G, returns = 0, []
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)

        # Normalize returns (variance reduction)
        R = np.array(returns, dtype=np.float64)
        if R.std() > 1e-6:
            R = (R - R.mean()) / R.std()

        policy.update(states, actions, R.tolist())
        episode_rewards.append(sum(rewards))

    return episode_rewards

print("Training REINFORCE agent (400 episodes)...")
episode_rewards = reinforce(env, policy, episodes=400, gamma=0.99, max_steps=100)
print(f"Done.")
print(f"  First 50 ep mean  : {np.mean(episode_rewards[:50]):>10.1f}")
print(f"  Ep 100-200 mean   : {np.mean(episode_rewards[100:200]):>10.1f}")
print(f"  Last 50 ep mean   : {np.mean(episode_rewards[-50:]):>10.1f}")
print(f"  All-time max      : {max(episode_rewards):>10.1f}")

## 7 · Convergence — The Gradient Ascending

REINFORCE converges more gradually than Q-Learning: there is no explicit exploration
schedule (no epsilon), and policy gradient estimates are high-variance.

Despite this, the cumulative reward per episode increases significantly:
from ~9,873 (first 50 episodes) to ~12,727 (last 50 episodes) —
a **+28.9% improvement** purely from policy gradient updates.

The variance in the reward curve reflects the stochastic nature of the environment
and the Monte Carlo return estimates. This is expected and can be reduced with
a baseline or Actor-Critic (the natural next step).


In [ ]:
# ── Convergence curve ────────────────────────────────────────────────────────
rewards_s = pd.Series(episode_rewards)
roll_50  = rewards_s.rolling(50).mean()
roll_100 = rewards_s.rolling(100).mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor(C_BG)

ax = axes[0]
ax.plot(episode_rewards, color=C_BLUE, lw=0.5, alpha=0.4, label="Episode reward")
ax.plot(roll_50,  color=C_WARN, lw=1.5, label="50-ep mean")
ax.plot(roll_100, color=C_OK,   lw=2.0, label="100-ep mean")
ax.set_xlabel("Episode"); ax.set_ylabel("Cumulative Episode Reward")
ax.set_title("REINFORCE Convergence", color="white", fontsize=12)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Learning phases bar chart
phases_names = ["Ep 1-50","Ep 50-150","Ep 150-250","Ep 250-350","Ep 350-400"]
phases_vals  = [np.mean(episode_rewards[:50]),
                np.mean(episode_rewards[50:150]),
                np.mean(episode_rewards[150:250]),
                np.mean(episode_rewards[250:350]),
                np.mean(episode_rewards[350:])]
ax2 = axes[1]
ax2.barh(phases_names, phases_vals, color=C_BLUE, edgecolor=C_BG, height=0.5)
for i, v in enumerate(phases_vals):
    ax2.text(v+50, i, f"{v:,.0f}", va="center", fontsize=9, color="#c9d1d9")
ax2.set_xlabel("Mean Episode Reward"); ax2.set_title("Learning Phases", color="white", fontsize=12)
ax2.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("fig_04_convergence.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()

print(f"First 50 ep mean : {np.mean(episode_rewards[:50]):,.0f}")
print(f"Last  50 ep mean : {np.mean(episode_rewards[-50:]):,.0f}")
print(f"Improvement      : +{np.mean(episode_rewards[-50:])-np.mean(episode_rewards[:50]):,.0f} ({(np.mean(episode_rewards[-50:])/np.mean(episode_rewards[:50])-1)*100:.1f}%)")

## 8 · Policy Analysis — What the Agent Learned

Two analytical views of the trained policy:

1. **Mean action probabilities** — the agent's learned preferences across states
2. **Policy weight matrix** — which state features drive each action decision

This interpretability layer converts the policy parameters into operational insight:
*why* does the agent prefer redirect-flow over decrease-speed?


In [ ]:
# ── Mean action probabilities across sampled states ──────────────────────────
sampled_probs = []
for _ in range(500):
    s = env.reset()
    sampled_probs.append(policy.probs(s))

mean_probs = np.mean(sampled_probs, axis=0)
std_probs  = np.std(sampled_probs, axis=0)

fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor(C_BG)

x = np.arange(6)
bars = ax.bar(x, mean_probs, color=C_BLUE, edgecolor=C_BG, alpha=0.85, label="Mean prob")
ax.errorbar(x, mean_probs, yerr=std_probs, fmt="none", color=C_WARN, capsize=5, lw=1.5)
ax.axhline(1/6, color=C_RED, lw=1.5, ls="--", label="Random baseline (1/6 ≈ 0.167)")
for bar, p in zip(bars, mean_probs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                        f"{p:.3f}", ha="center", fontsize=10, color="#c9d1d9")
ax.set_xticks(x); ax.set_xticklabels(ACTION_NAMES, rotation=15, ha="right", fontsize=9)
ax.set_ylabel("Mean Action Probability"); ax.set_ylim(0, 0.45)
ax.set_title("Trained Policy — Mean Action Probabilities (500 sampled states)",
             color="white", fontsize=11)
ax.legend(fontsize=9); ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("fig_05_action_probs.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()

print("Operational insight:")
for a_name, p, p_base in zip(ACTION_NAMES, mean_probs, [1/6]*6):
    diff = (p - p_base)*100
    direction = "▲ over-weighted" if diff > 2 else ("▼ under-weighted" if diff < -2 else "≈ neutral")
    print(f"  {a_name:<22} {p:.3f}  {direction}")

In [ ]:
# ── Policy weight heatmap — which state features drive each action? ───────────
feature_names = [f"CT-S{i}" for i in range(1,7)] + [
    "WIP", "Speed", "FailureP", "MicroSt", "OperEff"
]

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor(C_BG)

im = ax.imshow(policy.W, cmap="RdYlGn", aspect="auto")
plt.colorbar(im, ax=ax, label="Weight value (positive=action driver)", shrink=0.8)
ax.set_xticks(range(len(feature_names))); ax.set_xticklabels(feature_names, rotation=30, ha="right")
ax.set_yticks(range(6)); ax.set_yticklabels(ACTION_NAMES, fontsize=9)
ax.set_title("Policy Weight Matrix W — State Feature Influence per Action",
             color="white", fontsize=11, pad=10)
plt.tight_layout()
plt.savefig("fig_06_weights.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()

print("Green = feature pushes agent toward this action.")
print("Red   = feature pushes agent away from this action.")
print()
print("Key insights from weight matrix:")
print(f"  Redirect Flow weight on WIP       : {policy.W[4, 6]:.3f} (high WIP → redirect flow)")
print(f"  Quick Maint weight on FailureP    : {policy.W[3, 8]:.3f}")
print(f"  Reassign Op weight on OperEff     : {policy.W[0, 10]:.3f}")

## 9 · REINFORCE vs. Random — Performance Comparison

We compare the trained REINFORCE policy against the random baseline
on the same environment, same seeds. Key comparison metric: **mean step reward**.

Random baseline: **94.9 reward/step** (from `Data_people.csv`)
REINFORCE policy: mean step reward from a fresh evaluation run.


In [ ]:
# ── Evaluation: trained policy vs. random baseline ───────────────────────────
def evaluate_policy(env, policy, n_episodes=50, max_steps=100, random=False):
    # Evaluate a policy. If random=True, use uniform random actions.
    np.random.seed(99)
    all_step_rewards, all_throughputs = [], []
    for _ in range(n_episodes):
        s = env.reset()
        for _ in range(max_steps):
            a = (np.random.randint(env.action_space) if random
                 else policy.select_action(s))
            s, r, done, info = env.step(a)
            all_step_rewards.append(r)
            all_throughputs.append(info["throughput"])
            if done: break
    return np.mean(all_step_rewards), np.mean(all_throughputs)

reinforce_reward, reinforce_tp = evaluate_policy(env, policy, random=False)
random_reward,    random_tp    = evaluate_policy(env, policy, random=True)
baseline_reward = 94.86   # from Data_people.csv

print("=" * 55)
print(f"{'':25} {'Rand.Baseline':>12} {'REINFORCE':>12}")
print("=" * 55)
print(f"{'Mean Step Reward':<25} {random_reward:>12.2f} {reinforce_reward:>12.2f}")
print(f"{'Mean Throughput (UPH)':<25} {random_tp:>12.1f} {reinforce_tp:>12.1f}")
print(f"{'Improvement (reward)':<25} {'—':>12} {f'+{((reinforce_reward/random_reward)-1)*100:.1f}%':>12}")
print("=" * 55)

In [ ]:
# ── Side-by-side reward evolution in evaluation episodes ─────────────────────
def eval_episodes(env, policy, n=30, max_steps=100, random=False):
    np.random.seed(77)
    ep_rewards = []
    for _ in range(n):
        s = env.reset(); total = 0
        for _ in range(max_steps):
            a = np.random.randint(env.action_space) if random else policy.select_action(s)
            s, r, done, _ = env.step(a); total += r
            if done: break
        ep_rewards.append(total)
    return ep_rewards

rand_ep   = eval_episodes(env, policy, random=True)
reinf_ep  = eval_episodes(env, policy, random=False)

fig, ax = plt.subplots(figsize=(11, 4))
fig.patch.set_facecolor(C_BG)
x = range(len(rand_ep))
ax.plot(x, rand_ep,  color=C_RED,  lw=1.5, alpha=0.85, label="Random Policy")
ax.plot(x, reinf_ep, color=C_OK,   lw=1.5, alpha=0.85, label="REINFORCE Policy")
ax.axhline(np.mean(rand_ep),  color=C_RED,  lw=2, ls="--", alpha=0.6)
ax.axhline(np.mean(reinf_ep), color=C_OK,   lw=2, ls="--", alpha=0.6)
ax.fill_between(x, rand_ep, reinf_ep,
                where=[r < p for r, p in zip(rand_ep, reinf_ep)],
                alpha=0.15, color=C_OK, label="REINFORCE advantage")
ax.set_xlabel("Evaluation Episode"); ax.set_ylabel("Cumulative Episode Reward")
ax.set_title("REINFORCE vs. Random Policy — 30 Evaluation Episodes",
             color="white", fontsize=12)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fig_07_comparison.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()

## 10 · Statistical Validation

Two validation checks confirm the REINFORCE improvement is real and stable:

1. **Bootstrap confidence intervals** — is the mean reward difference statistically significant?
2. **Learning rate sensitivity** — does the improvement hold across hyperparameter choices?


In [ ]:
# ── Bootstrap CI for improvement ─────────────────────────────────────────────
np.random.seed(42)
n_boot = 2000

boot_rand   = [np.mean(np.random.choice(rand_ep,  len(rand_ep),  replace=True)) for _ in range(n_boot)]
boot_reinf  = [np.mean(np.random.choice(reinf_ep, len(reinf_ep), replace=True)) for _ in range(n_boot)]
boot_diff   = np.array(boot_reinf) - np.array(boot_rand)

ci_lo, ci_hi = np.percentile(boot_diff, [2.5, 97.5])
p_positive   = (boot_diff > 0).mean()

print("Bootstrap Confidence Interval — REINFORCE improvement over random:")
print(f"  Mean improvement : {np.mean(boot_diff):,.1f} reward/episode")
print(f"  95% CI           : [{ci_lo:,.1f},  {ci_hi:,.1f}]")
print(f"  P(REINFORCE > Random): {p_positive:.1%}")
print()
if ci_lo > 0:
    print("  Result: REINFORCE significantly outperforms random at 95% confidence.")
else:
    print("  Result: Improvement is not statistically significant at 95%.")

In [ ]:
# ── Learning rate sensitivity ─────────────────────────────────────────────────
print(f"{'LR':>8} {'Last 50 ep mean':>18} {'vs lr=0.002':>12}")
print("-" * 42)

for lr_test in [0.0005, 0.001, 0.002, 0.005, 0.01]:
    np.random.seed(42)
    p_test  = SoftmaxPolicy(state_dim=state_dim, n_actions=6, lr=lr_test)
    r_test  = reinforce(env, p_test, episodes=200)
    last_50 = np.mean(r_test[-50:])
    ref     = np.mean(episode_rewards[-50:])
    rel     = f"{(last_50/ref - 1)*100:+.1f}%"
    note    = "← used" if lr_test == 0.002 else ""
    print(f"{lr_test:>8.4f} {last_50:>18,.0f} {rel:>12}  {note}")

## 11 · Simulator — Deploy the Trained Policy

The simulator evaluates any assembly line state snapshot and returns
the policy's recommended action with its probability, plus the top-3 alternatives.


In [ ]:
# ── Policy simulator ─────────────────────────────────────────────────────────
def recommend_action(
    ct_norms,           # list of 6 normalized cycle times [0-1]
    wip_norm,           # WIP normalized [0-1]
    speed_norm,         # speed normalized [0-1]
    failure_prob,       # failure probability [0-0.25]
    micro_stops_norm,   # micro-stops normalized [0-1]
    operator_efficiency # operator efficiency [0.60-1.00]
):
    state = np.array(list(ct_norms) + [
        wip_norm, speed_norm, failure_prob,
        micro_stops_norm, operator_efficiency
    ], dtype=np.float32)

    probs = policy.probs(state)
    top3  = np.argsort(probs)[::-1][:3]

    print(f"Policy Recommendation:")
    print(f"  Top action : [{top3[0]}] {ACTION_NAMES[top3[0]]}  (p={probs[top3[0]]:.3f})")
    print(f"  Alternative: [{top3[1]}] {ACTION_NAMES[top3[1]]}  (p={probs[top3[1]]:.3f})")
    print(f"  Alternative: [{top3[2]}] {ACTION_NAMES[top3[2]]}  (p={probs[top3[2]]:.3f})")
    print()
    return top3[0], probs

In [ ]:
# ── Scenario A: High WIP, normal stations ────────────────────────────────────
print("=" * 55)
print("SCENARIO A — High WIP congestion, normal cycle times")
print("=" * 55)
recommend_action(
    ct_norms=[0.4]*6, wip_norm=0.75,
    speed_norm=0.78, failure_prob=0.05,
    micro_stops_norm=0.15, operator_efficiency=0.88
)

# ── Scenario B: High failure probability ──────────────────────────────────────
print("=" * 55)
print("SCENARIO B — High failure probability, many micro-stops")
print("=" * 55)
recommend_action(
    ct_norms=[0.5]*6, wip_norm=0.20,
    speed_norm=0.80, failure_prob=0.18,
    micro_stops_norm=0.70, operator_efficiency=0.82
)

# ── Scenario C: Low efficiency, slow stations ──────────────────────────────────
print("=" * 55)
print("SCENARIO C — Low operator efficiency, slow cycle times")
print("=" * 55)
recommend_action(
    ct_norms=[0.8]*6, wip_norm=0.25,
    speed_norm=0.72, failure_prob=0.07,
    micro_stops_norm=0.20, operator_efficiency=0.68
)

In [ ]:
# ── Final Reflection ──────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║   FINAL REFLECTION — REINFORCE & The Logic of Policy Gradient          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  Q-Learning remembered. REINFORCE decided.                              ║
║                                                                          ║
║  The distinction is not academic. A Q-table scales poorly with          ║
║  continuous states — each new dimension multiplies the table size.      ║
║  A policy function scales gracefully — it generalizes across states     ║
║  it has never seen, through the shared parameters W and b.              ║
║                                                                          ║
║  REINFORCE trained on this assembly line learned three things           ║
║  that a random manager never figures out:                               ║
║                                                                          ║
║    1. High WIP → redirect flow first, not increase speed                ║
║    2. High failure probability → maintenance beats any throughput gain  ║
║    3. Low operator efficiency → reassign before touching anything else  ║
║                                                                          ║
║  These are not rules that were programmed. They emerged from            ║
║  400 episodes of consequence — reward shaping what the gradient built.  ║
║                                                                          ║
║  One chapter remains: Model-Based RL.                                   ║
║  Where Q-Learning reacted and REINFORCE decided,                        ║
║  the model-based agent will plan — simulating futures before acting.    ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")
print("LozanoLsa | Operational Excellence · MBB · Machine Learning | GitHub: LozanoLsa")